
# Extraction des phrases où un personnage apparaît avec un mot cible

Ce notebook :

1. lit le fichier `.ods` des personnages ;
2. construit, pour chaque personnage, une liste de variantes de noms à partir des **3 premières colonnes** :
   - `NOMS HUMAINS`
   - `SECOND NOM`
   - `TROISIEME NOM`
3. prend en compte les combinaisons demandées :
   - `1 + 2`
   - `2 + 3`
   - `1 + 2 + 3`
   - `1 + 3`
4. repère dans le CSV les **phrases** qui contiennent au moins un mot cible (`mort`, `décéder`, `mourir`) ;
5. vérifie quels personnages apparaissent dans ces phrases ;
6. génère un tableau de sortie avec :
   - l'**identifiant unique** du personnage ;
   - tous ses **alias** ;
   - la **phrase** concernée.

## Logique de détection

- Les **mots cibles** sont recherchés dans la colonne `text_lemma` par défaut, pour repérer plus facilement les variantes fléchies.
- La **phrase exportée** provient de la colonne `text`.
- Les **noms de personnages** sont recherchés dans la phrase `text`.
- Une ligne = **une phrase pour un personnage donné**.
  Si deux personnages apparaissent dans la même phrase avec un mot cible, la phrase apparaîtra sur deux lignes.


In [1]:

# Paramètres
ODS_PATH = "Noms personnages de Pern Noms de famille.ods"
CSV_PATH = "La-ballade-de-Pern-intégrale-sentences-lemma.csv"
ODS_SHEET = "Nom et genre persos"

# True = compte aussi les noms simples (recommandé)
# False = ne compte QUE 1+2, 2+3, 1+2+3, 1+3
INCLUDE_SINGLE_NAMES = True

# Colonnes du CSV
TEXT_COLUMN = "text"           # phrase à exporter
LEMMA_COLUMN = "text_lemma"    # colonne utilisée pour repérer les mots cibles

# Mots cibles
TARGET_WORDS = ["mort", "décéder", "mourir"]

# Fichiers de sortie
OUTPUT_XLSX = "pern_personnages_mort_phrases.xlsx"
OUTPUT_CSV = "pern_personnages_mort_phrases.csv"
OUTPUT_ALIASES_CSV = "pern_aliases_utilises.csv"


In [2]:

import pandas as pd
import re
import unicodedata
from pathlib import Path


/home/marc-at/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/marc-at/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [4]:

# Lecture des fichiers
names_df = pd.read_excel(ODS_PATH, sheet_name=ODS_SHEET, engine="odf")
text_df = pd.read_csv(CSV_PATH)


In [5]:

# Vérifications minimales
required_name_cols = ["NOMS HUMAINS", "SECOND NOM", "TROISIEME NOM", "Id unique"]
required_csv_cols = [TEXT_COLUMN, LEMMA_COLUMN]

missing_ods = [c for c in required_name_cols if c not in names_df.columns]
missing_csv = [c for c in required_csv_cols if c not in text_df.columns]

if missing_ods:
    raise ValueError(f"Colonnes manquantes dans le ODS : {missing_ods}")

if missing_csv:
    raise ValueError(f"Colonnes manquantes dans le CSV : {missing_csv}")

print("Vérification OK.")


Vérification OK.


In [6]:

def clean_part(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


def normalize_text(s):
    """Minuscule + suppression des accents + espaces normalisés."""
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s


def build_aliases(row, include_single_names=True):
    """Construit les variantes d'un personnage à partir des 3 premières colonnes."""
    p1 = clean_part(row["NOMS HUMAINS"])
    p2 = clean_part(row["SECOND NOM"])
    p3 = clean_part(row["TROISIEME NOM"])

    aliases = []

    # Formes simples
    if include_single_names:
        for p in (p1, p2, p3):
            if p:
                aliases.append(p)

    # Combinaisons demandées
    if p1 and p2:
        aliases.append(f"{p1} {p2}")      # 1 + 2
    if p2 and p3:
        aliases.append(f"{p2} {p3}")      # 2 + 3
    if p1 and p2 and p3:
        aliases.append(f"{p1} {p2} {p3}") # 1 + 2 + 3
    if p1 and p3:
        aliases.append(f"{p1} {p3}")      # 1 + 3

    # Déduplication en conservant l'ordre
    seen = set()
    clean_aliases = []
    for a in aliases:
        a = re.sub(r"\s+", " ", a).strip()
        key = normalize_text(a)
        if a and key not in seen:
            seen.add(key)
            clean_aliases.append(a)

    # Les alias longs d'abord pour mieux repérer les formes composées
    clean_aliases = sorted(clean_aliases, key=lambda x: (-len(x), normalize_text(x)))
    return clean_aliases


names_df = names_df.copy()
names_df["aliases"] = names_df.apply(
    lambda row: build_aliases(row, include_single_names=INCLUDE_SINGLE_NAMES),
    axis=1
)

names_df[["NOMS HUMAINS", "SECOND NOM", "TROISIEME NOM", "Id unique", "aliases"]].head(15)


,NOMS HUMAINS,SECOND NOM,TROISIEME NOM,Id unique,aliases
0,Kenjo,Fusaiyuki,NaN,1.0,"[Kenjo Fusaiyuki, Fusaiyuki, Kenjo]"
1,Abula,NaN,NaN,2.0,[Abula]
2,Abuna,NaN,NaN,3.0,[Abuna]
3,Adessa,NaN,NaN,4.0,[Adessa]
4,Adrea,Beljeth,NaN,5.0,"[Adrea Beljeth, Beljeth, Adrea]"
5,Aida,NaN,NaN,6.0,[Aida]
6,Alda,NaN,NaN,7.0,[Alda]
7,Alianne,Zulueta,NaN,8.0,"[Alianne Zulueta, Alianne, Zulueta]"
8,Aliena,Zuleita,NaN,9.0,"[Aliena Zuleita, Zuleita, Aliena]"
9,Almie,NaN,NaN,10.0,[Almie]


In [7]:

# Tableau d'audit des alias réellement utilisés
aliases_audit = names_df[["Id unique", "NOMS HUMAINS", "SECOND NOM", "TROISIEME NOM", "aliases"]].copy()
aliases_audit["aliases"] = aliases_audit["aliases"].apply(lambda x: " | ".join(x))
aliases_audit.to_csv(OUTPUT_ALIASES_CSV, index=False)
print(f"Fichier alias exporté : {OUTPUT_ALIASES_CSV}")
aliases_audit.head(10)


Fichier alias exporté : pern_aliases_utilises.csv


,Id unique,NOMS HUMAINS,SECOND NOM,TROISIEME NOM,aliases
0,1.0,Kenjo,Fusaiyuki,NaN,Kenjo Fusaiyuki | Fusaiyuki | Kenjo
1,2.0,Abula,NaN,NaN,Abula
2,3.0,Abuna,NaN,NaN,Abuna
3,4.0,Adessa,NaN,NaN,Adessa
4,5.0,Adrea,Beljeth,NaN,Adrea Beljeth | Beljeth | Adrea
5,6.0,Aida,NaN,NaN,Aida
6,7.0,Alda,NaN,NaN,Alda
7,8.0,Alianne,Zulueta,NaN,Alianne Zulueta | Alianne | Zulueta
8,9.0,Aliena,Zuleita,NaN,Aliena Zuleita | Zuleita | Aliena
9,10.0,Almie,NaN,NaN,Almie


In [8]:

def compile_character_pattern(aliases):
    """Crée une regex qui repère les alias du personnage avec bornes souples."""
    if not aliases:
        return None

    escaped = [re.escape(normalize_text(a)) for a in aliases if a]
    if not escaped:
        return None

    pattern = r"(?<!\w)(?:" + "|".join(escaped) + r")(?!\w)"
    return re.compile(pattern, flags=re.IGNORECASE)


def compile_target_pattern(target_words):
    normalized_targets = [re.escape(normalize_text(w)) for w in target_words if w]
    pattern = r"(?<!\w)(?:" + "|".join(normalized_targets) + r")(?!\w)"
    return re.compile(pattern, flags=re.IGNORECASE)


def find_all_matches(pattern, text):
    if pattern is None:
        return []
    return pattern.findall(text)


In [9]:

# Préparation du CSV
text_df = text_df.copy()
text_df[TEXT_COLUMN] = text_df[TEXT_COLUMN].fillna("").astype(str)
text_df[LEMMA_COLUMN] = text_df[LEMMA_COLUMN].fillna("").astype(str)

text_df["text_norm"] = text_df[TEXT_COLUMN].apply(normalize_text)
text_df["lemma_norm"] = text_df[LEMMA_COLUMN].apply(normalize_text)

target_pattern = compile_target_pattern(TARGET_WORDS)
text_df["target_matches"] = text_df["lemma_norm"].apply(lambda x: sorted(set(find_all_matches(target_pattern, x))))
text_df["has_target_word"] = text_df["target_matches"].apply(bool)

candidate_df = text_df[text_df["has_target_word"]].copy()

print(f"Nombre total de phrases : {len(text_df):,}")
print(f"Nombre de phrases contenant un mot cible : {len(candidate_df):,}")

candidate_df[["sentence_id", TEXT_COLUMN, LEMMA_COLUMN, "target_matches"]].head(10)


Nombre total de phrases : 132,850
Nombre de phrases contenant un mot cible : 1,012


,sentence_id,text,text_lemma,target_matches
61,62,Il préférait prendre des décisions rapides et ...,Il préférer prendre un décision rapide et le m...,[mort]
487,488,La mort de ses parents lui avait donné le droi...,Le mort de son parent luire avoir donner le dr...,[mort]
769,770,"Elle parlait avec douceur, sachant que Kenjo a...","Lui parler avec douceur, savoir que Kenjo avoi...",[mort]
900,901,"Personne n’avait jamais posé le pied sur Pern,...",Personne n’avoir jamais poser le pied sur Pern...,[mort]
925,926,"La navette atterrirait sans problème, mais l’a...","Le navette atterrir sans problème, mais l’amir...",[mourir]
1896,1897,"— Il meurt de faim, Sean, dit Sorka, cherchant...","— Il mourir de faim, Sean, dire Sorka, cherche...",[mourir]
2087,2088,— C’est parce qu’il mourait de faim après l’éc...,— C’être parce qu’il mourir de faim après l’éc...,[mourir]
2141,2142,"Les zoologues, qui en mouraient d’envie, n’ava...","Le zoologue, qui en mourir d’envie, n’avoir pa...",[mourir]
2510,2511,Le tremblement de terre dura le temps d’un bat...,Le tremblement de terre durer le temps d’un ba...,[mort]
2871,2872,Et seulement un nouveau-né mort des autres cou...,Et seulement un nouveau-naître mort de autre c...,[mort]


In [10]:

# Extraction des phrases par personnage
results = []

for i, row in names_df.iterrows():
    char_id = row["Id unique"]
    aliases = row["aliases"]
    aliases_str = " | ".join(aliases)
    char_pattern = compile_character_pattern(aliases)

    if char_pattern is None:
        continue

    char_matches_mask = candidate_df["text_norm"].str.contains(char_pattern, na=False)
    matched_rows = candidate_df[char_matches_mask]

    for _, sent_row in matched_rows.iterrows():
        matched_aliases = sorted(set(find_all_matches(char_pattern, sent_row["text_norm"])))
        results.append({
            "Id unique": char_id,
            "aliases": aliases_str,
            "phrase": sent_row[TEXT_COLUMN],
            "sentence_id": sent_row.get("sentence_id", pd.NA),
            "book": sent_row.get("book", pd.NA),
            "mots_cibles": " | ".join(sent_row["target_matches"]),
            "alias_trouves_dans_phrase": " | ".join(matched_aliases),
        })

    if (i + 1) % 100 == 0:
        print(f"{i + 1} personnages traités / {len(names_df)}")

result_df = pd.DataFrame(results)
print("Extraction terminée.")
print("Nombre de lignes extraites :", len(result_df))
result_df.head(10)


100 personnages traités / 1067
200 personnages traités / 1067
300 personnages traités / 1067
400 personnages traités / 1067
500 personnages traités / 1067
600 personnages traités / 1067
700 personnages traités / 1067
800 personnages traités / 1067
900 personnages traités / 1067
1000 personnages traités / 1067
Extraction terminée.
Nombre de lignes extraites : 1849


,Id unique,aliases,phrase,sentence_id,book,mots_cibles,alias_trouves_dans_phrase
0,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,"Elle parlait avec douceur, sachant que Kenjo a...",770,1.0,mort,kenjo
1,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,"— Kenjo est mort et Ongola grièvement blessé, ...",6063,1.0,mort,kenjo
2,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,"Au Terminus, la mort de Kenjo et l’état grave ...",6554,1.0,mort,kenjo
3,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,J’ai trouvé ses initiales à plusieurs endroits...,120411,15.0,mort,fusaiyuki | kenjo
4,4.0,Adessa,Ne me dis pas que cette chère Dame Adessa est ...,60657,7.0,mourir,adessa
5,14.0,Anella,— Tolocamp a mis Anella dans son lit une heure...,39884,5.0,mort,anella
6,15.0,Aramina Mina Ara | Aramina Mina | Aramina Ara ...,Furieuse qu’on ait laissé croire à Benden qu’A...,48045,6.0,mourir,aramina
7,15.0,Aramina Mina Ara | Aramina Mina | Aramina Ara ...,"Aramina avait été sauvée, et elle vivait saine...",48137,6.0,mourir,aramina
8,15.0,Aramina Mina Ara | Aramina Mina | Aramina Ara ...,Il avait souvent assuré à Ara que cette femme ...,48967,6.0,mort,ara
9,15.0,Aramina Mina Ara | Aramina Mina | Aramina Ara ...,"— Tous les chiens sont morts, Ara ?",49164,6.0,mort,ara


In [11]:

# Nettoyage final : une ligne = une phrase pour un personnage
if not result_df.empty:
    result_df = result_df.drop_duplicates(subset=["Id unique", "phrase"]).copy()

    # Tri pour lecture plus facile
    sort_cols = [c for c in ["Id unique", "book", "sentence_id"] if c in result_df.columns]
    result_df = result_df.sort_values(sort_cols).reset_index(drop=True)

print(result_df.shape)
result_df.head(20)


(1734, 7)


,Id unique,aliases,phrase,sentence_id,book,mots_cibles,alias_trouves_dans_phrase
0,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,"Elle parlait avec douceur, sachant que Kenjo a...",770,1.0,mort,kenjo
1,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,"— Kenjo est mort et Ongola grièvement blessé, ...",6063,1.0,mort,kenjo
2,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,"Au Terminus, la mort de Kenjo et l’état grave ...",6554,1.0,mort,kenjo
3,1.0,Kenjo Fusaiyuki | Fusaiyuki | Kenjo,J’ai trouvé ses initiales à plusieurs endroits...,120411,15.0,mort,fusaiyuki | kenjo
4,4.0,Adessa,Ne me dis pas que cette chère Dame Adessa est ...,60657,7.0,mourir,adessa
5,14.0,Anella,— Tolocamp a mis Anella dans son lit une heure...,39884,5.0,mort,anella
6,15.0,Aramina Mina Ara | Aramina Mina | Aramina Ara ...,Furieuse qu’on ait laissé croire à Benden qu’A...,48045,6.0,mourir,aramina
7,15.0,Aramina Mina Ara | Aramina Mina | Aramina Ara ...,"Aramina avait été sauvée, et elle vivait saine...",48137,6.0,mourir,aramina
8,15.0,Aramina Mina Ara | Aramina Mina | Aramina Ara ...,Il avait souvent assuré à Ara que cette femme ...,48967,6.0,mort,ara
9,15.0,Aramina Mina Ara | Aramina Mina | Aramina Ara ...,"— Tous les chiens sont morts, Ara ?",49164,6.0,mort,ara


In [12]:

# Export
result_df.to_excel(OUTPUT_XLSX, index=False)
result_df.to_csv(OUTPUT_CSV, index=False)

print(f"Fichier Excel exporté : {OUTPUT_XLSX}")
print(f"Fichier CSV exporté   : {OUTPUT_CSV}")


Fichier Excel exporté : pern_personnages_mort_phrases.xlsx
Fichier CSV exporté   : pern_personnages_mort_phrases.csv
